# Baseline Model 1

Features:
- Chromosome
- ref_length
- alt_length
- length_difference
- mutation_type

Results:
- Accuracy: 86%
- F1 pathogenic: 0.52

In [ ]:
import pandas as pd

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

In [ ]:
df = pd.read_csv("../data/processed/clinvar_features.csv")
df.shape

In [ ]:
sample_df=df.sample(n=10000, random_state=42).copy()

In [ ]:
le_chr = LabelEncoder()
le_mut = LabelEncoder()

sample_df["Chromosome_encoded"] = le_chr.fit_transform(
    sample_df["Chromosome"].astype(str)
)

sample_df["mutation_type_encoded"] = le_mut.fit_transform(
    sample_df["mutation_type"].astype(str)
)

In [ ]:
X = sample_df[
    [
        "Chromosome_encoded",
        "ref_length",
        "alt_length",
        "length_difference",
        "mutation_type_encoded"
    ]
]

y = sample_df["target"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
rf_baseline_1 = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

rf_baseline_1.fit(X_train, y_train)

In [ ]:
predictions_1 = rf_baseline_1.predict(X_test)

print(classification_report(y_test, predictions_1))

# BASELINE MODEL 2

# Features: Chromosome + GeneSymbol + mutation features

# Encoding: LabelEncoder

# Model: Random Forest

Addition of GeneSymbol using Label Encoding decreased overall model performance. Although GeneSymbol_encoded received the highest feature importance score (57.8%), the F1-score for pathogenic variants decreased from 0.53 to 0.44. This suggests that Label Encoding introduced artificial ordinal relationships between genes, leading to poorer generalization.

In [ ]:
le_gene = LabelEncoder()

sample_df["GeneSymbol_encoded"] = le_gene.fit_transform(
    sample_df["GeneSymbol"].astype(str)
)

In [ ]:
X = sample_df[
    [
        "Chromosome_encoded",
        "GeneSymbol_encoded",
        "ref_length",
        "alt_length",
        "length_difference",
        "mutation_type_encoded"
    ]
]

y = sample_df["target"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
rf_baseline_2 = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

rf_baseline_2.fit(X_train, y_train)

In [ ]:
predictions_2 = rf_baseline_2.predict(X_test)

print(classification_report(y_test, predictions_2))

In [ ]:
feature_importances = pd.DataFrame({
    "feature": X.columns,
    "importance": rf_baseline_2.feature_importances_
}).sort_values(by="importance", ascending=False)

feature_importances

# Baseline Model 3

## Objective

Evaluate whether GeneSymbol improves variant classification when encoded using One-Hot Encoding instead of Label Encoding.

### Features

- Chromosome
- GeneSymbol
- ref_length
- alt_length
- length_difference
- mutation_type

### Model

Random Forest Classifier

### Motivation

In Baseline Model 2, Label Encoding introduced artificial ordinal relationships between genes and reduced performance.

This experiment uses One-Hot Encoding to represent genes as independent categorical variables.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [ ]:
features = [
    "Chromosome",
    "GeneSymbol",
    "mutation_type",
    "ref_length",
    "alt_length",
    "length_difference"
]

target = "target"

In [ ]:
for col in ["Chromosome", "GeneSymbol", "mutation_type"]:
    sample_df[col] = sample_df[col].astype(str)

In [ ]:
X = sample_df[features]
y = sample_df[target]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
categorical_features = [
    "Chromosome",
    "GeneSymbol",
    "mutation_type"
]

numeric_features = [
    "ref_length",
    "alt_length",
    "length_difference"
]

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

In [ ]:
rf_baseline_3 = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=120,
            random_state=42,
            n_jobs=1,
            class_weight="balanced"
        ))
    ]
)

In [ ]:
rf_baseline_3.fit(X_train, y_train)

In [ ]:
predictions_3 = rf_baseline_3.predict(X_test)
print(classification_report(y_test, predictions_3))

# Baseline Model 4

This model extends Baseline Model 3 by incorporating additional clinical information available in ClinVar.

### Additional Features

- OriginSimple
- ReviewStatus

The objective is to evaluate whether clinical context improves variant pathogenicity prediction compared to models based only on genomic characteristics.

In [ ]:
features = [
    "Chromosome",
    "GeneSymbol",
    "mutation_type",
    "OriginSimple",
    "ReviewStatus",
    "ref_length",
    "alt_length",
    "length_difference"
]

target = "target"

In [ ]:
df_model4 = df[features + [target]].dropna().copy()
df_model4.shape

In [ ]:
categorical_features = [
    "Chromosome",
    "GeneSymbol",
    "mutation_type",
    "OriginSimple",
    "ReviewStatus"
]

numeric_features = [
    "ref_length",
    "alt_length",
    "length_difference"
]

for col in categorical_features:
    df_model4[col] = df_model4[col].astype(str)

In [ ]:
sample_model4 = df_model4.sample(
    n=10000,
    random_state=42
).copy()

sample_model4.shape

In [ ]:
X = sample_model4[features]
y = sample_model4[target]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "num",
            "passthrough",
            numeric_features
        )
    ]
)

In [ ]:
rf_baseline_4 = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=120,
            random_state=42,
            n_jobs=-1,
            class_weight="balanced"
        ))
    ]
)

In [ ]:
rf_baseline_4.fit(X_train, y_train)

In [ ]:
predictions_4 = rf_baseline_4.predict(X_test)

print(classification_report(y_test, predictions_4))